In [ ]:
# ============================================
# DIMONIC'S KAGGLE DATASET SPLITTER - FINAL VERSION
# ============================================

import os
import shutil
import random
from tqdm import tqdm
import zipfile
from pathlib import Path

# ===== PATH KONFIGURASI =====
INPUT_PATH = "/kaggle/input/datasets/yahyaazz/dataset-skripsi/penyakit daun padi"
OUTPUT_PATH = "/kaggle/working/"

# Tujuan finalnya (di dalam /kaggle/working/data)
FINAL_DATA_PATH = os.path.join(OUTPUT_PATH, "data")

print("="*60)
print("🔥 DIMONIC'S DATASET SPLITTER - KAGGLE EDITION 🔥")
print("="*60)
print(f"📂 INPUT PATH  : {INPUT_PATH}")
print(f"📂 OUTPUT PATH : {FINAL_DATA_PATH}")
print("="*60)

def split_dataset_11_kelas(sumber_dir, tujuan_dir, rasio_train=0.8, seed=42):
    """
    Fungsi khusus buat 11 kelas penyakit daun padi
    Output: data/train/kelas & data/test/kelas
    """
    
    # Set random seed biar konsisten
    random.seed(seed)
    
    # Setup folder train dan test
    train_dir = os.path.join(tujuan_dir, 'train')
    test_dir = os.path.join(tujuan_dir, 'test')
    
    # Bersihin folder tujuan kalo udah ada
    if os.path.exists(tujuan_dir):
        print("⚠️  Folder data udah ada, bakal ditimpa...")
        shutil.rmtree(tujuan_dir)
    
    # Bikin folder structure
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    # Cek apakah input path ada
    if not os.path.exists(sumber_dir):
        print(f"❌ ERROR: Path input gak ketemu!")
        print(f"   {sumber_dir}")
        print("\n📌 Isi dari /kaggle/input/datasets/yahyaazz/dataset-skripsi/:")
        parent_path = "/kaggle/input/datasets/yahyaazz/dataset-skripsi/"
        if os.path.exists(parent_path):
            print(os.listdir(parent_path))
        else:
            print("Path parent juga gak ketemu :(")
        return None
    
    # List semua folder di dalam direktori sumber
    print("\n🔍 Scanning folder di dalam input path...")
    all_items = os.listdir(sumber_dir)
    
    # Filter cuma folder aja (11 folder penyakit)
    kelas_kelas = []
    for item in all_items:
        item_path = os.path.join(sumber_dir, item)
        if os.path.isdir(item_path):
            kelas_kelas.append(item)
    
    if not kelas_kelas:
        # Mungkin filenya langsung di root tanpa folder?
        print("⚠️  Gak nemu folder, mungkin struktur datasetnya beda?")
        print(f"Isi dari {sumber_dir}:")
        for item in all_items[:10]:  # Show first 10
            print(f"   - {item}")
        return None
    
    print(f"\n✅ Nemu {len(kelas_kelas)} kelas penyakit:")
    for i, kelas in enumerate(kelas_kelas, 1):
        print(f"   {i}. {kelas}")
    
    # Kalo kurang dari 11, kasih warning tapi tetep lanjut
    if len(kelas_kelas) != 11:
        print(f"\n⚠️  Warning: Yang ada cuma {len(kelas_kelas)} folder (bukan 11)")
        print("   Tapi gapapa, tetep diproses kok!")
    
    print("\n" + "="*60)
    print("🔄 MULAI PROSES SPLIT DATASET...")
    print("="*60)
    
    # Statistik
    total_train = 0
    total_test = 0
    detail_kelas = {}
    
    # Proses setiap kelas pake progress bar
    for kelas in tqdm(kelas_kelas, desc="📊 Progress Split"):
        path_kelas = os.path.join(sumber_dir, kelas)
        
        # Kumpulin semua file gambar
        gambar = []
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            gambar.extend([f for f in os.listdir(path_kelas) if f.endswith(ext)])
        
        jumlah_gambar = len(gambar)
        if jumlah_gambar == 0:
            print(f"\n⚠️  Kelas {kelas} gak ada gambar, skip...")
            continue
        
        # Acak gambarnya
        random.shuffle(gambar)
        
        # Hitung batas train (80%)
        batas_train = int(jumlah_gambar * rasio_train)
        gambar_train = gambar[:batas_train]
        gambar_test = gambar[batas_train:]
        
        # Buat folder kelas di train dan test
        train_kelas_dir = os.path.join(train_dir, kelas)
        test_kelas_dir = os.path.join(test_dir, kelas)
        os.makedirs(train_kelas_dir, exist_ok=True)
        os.makedirs(test_kelas_dir, exist_ok=True)
        
        # Copy file ke train
        for img in gambar_train:
            src = os.path.join(path_kelas, img)
            dst = os.path.join(train_kelas_dir, img)
            shutil.copy2(src, dst)
        
        # Copy file ke test
        for img in gambar_test:
            src = os.path.join(path_kelas, img)
            dst = os.path.join(test_kelas_dir, img)
            shutil.copy2(src, dst)
        
        # Simpen statistik
        detail_kelas[kelas] = {
            'total': jumlah_gambar,
            'train': len(gambar_train),
            'test': len(gambar_test)
        }
        total_train += len(gambar_train)
        total_test += len(gambar_test)
        
        # Print detail per kelas (pake \r biar rapi)
        print(f"\n  📁 {kelas}: {jumlah_gambar:3d} gambar → train: {len(gambar_train):3d} | test: {len(gambar_test):3d}")
    
    # ===== LAPORAN AKHIR =====
    print("\n" + "="*60)
    print("✅✅✅ SPLIT DATASET SELESAI! ✅✅✅")
    print("="*60)
    print("\n📊 RINGKASAN STATISTIK:")
    print("-"*40)
    print(f"📁 Total kelas        : {len(kelas_kelas)}")
    print(f"🖼️  Total gambar       : {total_train + total_test}")
    print(f"🎓 Data Train (80%)    : {total_train} gambar")
    print(f"🧪 Data Test  (20%)    : {total_test} gambar")
    print(f"📊 Rasio Train:Test    : {total_train/(total_train+total_test)*100:.1f}% : {total_test/(total_train+total_test)*100:.1f}%")
    
    print("\n📋 DETAIL PER KELAS:")
    print("-"*60)
    print(f"{'No':<4} {'Kelas':<20} {'Total':<8} {'Train':<8} {'Test':<8}")
    print("-"*60)
    
    for i, (kelas, stat) in enumerate(detail_kelas.items(), 1):
        print(f"{i:<4} {kelas[:20]:<20} {stat['total']:<8} {stat['train']:<8} {stat['test']:<8}")
    
    print("-"*60)
    
    # ===== VERIFIKASI STRUKTUR =====
    print("\n🔍 VERIFIKASI STRUKTUR FOLDER:")
    print("-"*40)
    
    # Cek folder train
    train_folders = os.listdir(train_dir) if os.path.exists(train_dir) else []
    print(f"\n📁 train/ ({len(train_folders)} folder):")
    for folder in sorted(train_folders)[:5]:  # Show first 5
        folder_path = os.path.join(train_dir, folder)
        if os.path.isdir(folder_path):
            img_count = len([f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            print(f"   ├─ {folder}/ ({img_count} gambar)")
    
    if len(train_folders) > 5:
        print(f"   └─ ... dan {len(train_folders)-5} folder lainnya")
    
    # Cek folder test
    test_folders = os.listdir(test_dir) if os.path.exists(test_dir) else []
    print(f"\n📁 test/ ({len(test_folders)} folder):")
    for folder in sorted(test_folders)[:5]:
        folder_path = os.path.join(test_dir, folder)
        if os.path.isdir(folder_path):
            img_count = len([f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            print(f"   ├─ {folder}/ ({img_count} gambar)")
    
    if len(test_folders) > 5:
        print(f"   └─ ... dan {len(test_folders)-5} folder lainnya")
    
    # ===== CREATE ZIP FILE =====
    print("\n📦 MENG-ZIP HASIL...")
    zip_path = os.path.join(OUTPUT_PATH, 'dataset_penyakit_padi_split.zip')
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(tujuan_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, os.path.dirname(tujuan_dir))
                zipf.write(file_path, arcname)
    
    # ===== CREATE README =====
    readme_path = os.path.join(tujuan_dir, "README.txt")
    with open(readme_path, 'w') as f:
        f.write("="*50 + "\n")
        f.write("DATASET PENYAKIT DAUN PADI\n")
        f.write("="*50 + "\n\n")
        f.write(f"Total gambar: {total_train + total_test}\n")
        f.write(f"Train (80%): {total_train} gambar\n")
        f.write(f"Test (20%): {total_test} gambar\n")
        f.write(f"Jumlah kelas: {len(kelas_kelas)}\n\n")
        f.write("Daftar kelas:\n")
        for kelas in sorted(kelas_kelas):
            f.write(f"- {kelas}\n")
    
    print("\n" + "="*60)
    print("🎉🎉🎉 SELAMAT! DATASET LO UDAH SIAP! 🎉🎉🎉")
    print("="*60)
    print(f"\n📁 Hasil split ada di: {FINAL_DATA_PATH}")
    print(f"   ├─ train/ ({total_train} gambar)")
    print(f"   └─ test/ ({total_test} gambar)")
    print(f"\n📦 File zip: {zip_path}")
    print(f"📝 README: {readme_path}")
    print("\n💡 Next step: Lo bisa langsung pake buat training!")
    print("   Contoh: train_generator = flow_from_directory('data/train', ...)")
    print("\n" + "="*60)
    
    return detail_kelas

# ===== JALANKAN SCRIPT =====
if __name__ == "__main__":
    hasil_split = split_dataset_11_kelas(
        sumber_dir=INPUT_PATH,
        tujuan_dir=FINAL_DATA_PATH,
        rasio_train=0.8,
        seed=42
    )
    
    # Kalo hasil_split None berarti error
    if hasil_split is None:
        print("\n❌ Gagal! Cek lagi path inputnya bro!")
        print("Mungkin struktur foldernya beda? Coba manual:")
        print("!ls /kaggle/input/datasets/yahyaazz/dataset-skripsi/")
        

In [ ]:
# ============================================
# DIMONIC'S DATASET INSPECTOR - CHECK HASIL SPLIT
# ============================================

import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter
import numpy as np
from pathlib import Path

# ===== PATH DATASET HASIL SPLIT =====
DATA_PATH = "/kaggle/working/data"  # Folder data hasil split
TRAIN_PATH = os.path.join(DATA_PATH, "train")
TEST_PATH = os.path.join(DATA_PATH, "test")

print("="*70)
print("🔎 DIMONIC'S DATASET INSPECTOR - CHECK HASIL SPLIT 🔍")
print("="*70)

def cek_struktur_folder(path, level=0, max_items=5):
    """Cek struktur folder secara visual"""
    if not os.path.exists(path):
        print(f"❌ Folder {path} gak ketemu!")
        return
    
    indent = "   " * level
    items = sorted([f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))])
    
    if level == 0:
        print(f"\n📁 {os.path.basename(path)}/ ({len(items)} folder)")
    
    for i, item in enumerate(items[:max_items]):
        item_path = os.path.join(path, item)
        sub_items = [f for f in os.listdir(item_path) if os.path.isdir(os.path.join(item_path, f))]
        
        # Hitung jumlah gambar di folder ini
        gambar = [f for f in os.listdir(item_path) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        if level == 0:
            print(f"{indent}   ├─ 📂 {item}/ ({len(gambar)} gambar)")
        else:
            print(f"{indent}   ├─ {item}/ ({len(gambar)} gambar)")
        
        # Rekursif untuk subfolder (tapi kita limit depth)
        if level < 1 and sub_items:
            cek_struktur_folder(item_path, level+1, max_items)
    
    if len(items) > max_items:
        print(f"{indent}   └─ ... dan {len(items) - max_items} folder lainnya")

def hitung_statistik_dataset(train_path, test_path):
    """Hitung statistik lengkap dataset"""
    
    print("\n📊 STATISTIK DATASET:")
    print("-"*70)
    
    # Cek apakah path ada
    if not os.path.exists(train_path):
        print(f"❌ Train path gak ketemu: {train_path}")
        return None
    
    if not os.path.exists(test_path):
        print(f"❌ Test path gak ketemu: {test_path}")
        return None
    
    # List kelas dari train folder
    kelas_train = sorted([f for f in os.listdir(train_path) 
                         if os.path.isdir(os.path.join(train_path, f))])
    
    kelas_test = sorted([f for f in os.listdir(test_path) 
                        if os.path.isdir(os.path.join(test_path, f))])
    
    print(f"\n📁 Jumlah kelas di train: {len(kelas_train)}")
    print(f"📁 Jumlah kelas di test: {len(kelas_test)}")
    
    # Cek konsistensi kelas
    if set(kelas_train) == set(kelas_test):
        print("✅ Kelas train dan test KONSISTEN!")
    else:
        print("⚠️  Kelas train dan test BERBEDA!")
        print(f"   Kelas cuma di train: {set(kelas_train) - set(kelas_test)}")
        print(f"   Kelas cuma di test: {set(kelas_test) - set(kelas_train)}")
    
    # Hitung detail per kelas
    print("\n📋 DETAIL PER KELAS:")
    print("-"*90)
    print(f"{'No':<4} {'Nama Kelas':<25} {'Train':<12} {'Test':<12} {'Total':<12} {'Rasio Train/Test':<15}")
    print("-"*90)
    
    total_train_images = 0
    total_test_images = 0
    detail_data = {}
    
    for i, kelas in enumerate(sorted(set(kelas_train + kelas_test)), 1):
        # Hitung train images
        train_kelas_path = os.path.join(train_path, kelas) if kelas in kelas_train else None
        if train_kelas_path and os.path.exists(train_kelas_path):
            train_images = len([f for f in os.listdir(train_kelas_path) 
                              if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            train_images = 0
        
        # Hitung test images
        test_kelas_path = os.path.join(test_path, kelas) if kelas in kelas_test else None
        if test_kelas_path and os.path.exists(test_kelas_path):
            test_images = len([f for f in os.listdir(test_kelas_path) 
                             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            test_images = 0
        
        total = train_images + test_images
        total_train_images += train_images
        total_test_images += test_images
        
        # Hitung rasio
        if total > 0:
            rasio = f"{train_images/total*100:.1f}% : {test_images/total*100:.1f}%"
        else:
            rasio = "0 : 0"
        
        detail_data[kelas] = {
            'train': train_images,
            'test': test_images,
            'total': total
        }
        
        print(f"{i:<4} {kelas[:25]:<25} {train_images:<12} {test_images:<12} {total:<12} {rasio:<15}")
    
    print("-"*90)
    print(f"{'':<29} {total_train_images:<12} {total_test_images:<12} {total_train_images+total_test_images:<12}")
    
    # Statistik tambahan
    print("\n📈 STATISTIK TAMBAHAN:")
    print("-"*70)
    
    # Rata-rata gambar per kelas
    avg_train = total_train_images / len(detail_data) if detail_data else 0
    avg_test = total_test_images / len(detail_data) if detail_data else 0
    avg_total = (total_train_images + total_test_images) / len(detail_data) if detail_data else 0
    
    print(f"🎯 Rata-rata gambar per kelas:")
    print(f"   - Train: {avg_train:.1f} gambar")
    print(f"   - Test: {avg_test:.1f} gambar")
    print(f"   - Total: {avg_total:.1f} gambar")
    
    # Kelas dengan gambar paling banyak & sedikit
    if detail_data:
        kelas_terbanyak = max(detail_data.items(), key=lambda x: x[1]['total'])
        kelas_tersedikit = min(detail_data.items(), key=lambda x: x[1]['total'])
        
        print(f"\n🏆 Kelas dengan gambar TERBANYAK:")
        print(f"   - {kelas_terbanyak[0]}: {kelas_terbanyak[1]['total']} gambar")
        print(f"\n📉 Kelas dengan gambar TERSEDIKIT:")
        print(f"   - {kelas_tersedikit[0]}: {kelas_tersedikit[1]['total']} gambar")
        
        # Cek imbalance
        max_total = kelas_terbanyak[1]['total']
        min_total = kelas_tersedikit[1]['total']
        if min_total > 0:
            rasio_imbalance = max_total / min_total
            print(f"\n⚖️  Rasio imbalance (max/min): {rasio_imbalance:.2f}x")
            if rasio_imbalance > 2:
                print("   ⚠️  Warning: Dataset agak imbalance nih!")
            else:
                print("   ✅ Dataset cukup balance!")
    
    return detail_data

def preview_random_images(class_path, num_samples=4, class_name=""):
    """Preview random images dari suatu kelas"""
    if not os.path.exists(class_path):
        return
    
    images = [f for f in os.listdir(class_path) 
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if not images:
        return
    
    # Pilih random images
    samples = np.random.choice(images, min(num_samples, len(images)), replace=False)
    
    fig, axes = plt.subplots(1, len(samples), figsize=(15, 4))
    if len(samples) == 1:
        axes = [axes]
    
    for i, img_name in enumerate(samples):
        img_path = os.path.join(class_path, img_name)
        img = mpimg.imread(img_path)
        
        axes[i].imshow(img)
        axes[i].set_title(f"{class_name[:15]}\n{img_name[:10]}...", fontsize=10)
        axes[i].axis('off')
    
    plt.suptitle(f"🎲 Random Sample - {class_name}", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()

def cek_dataset_komplit():
    """Main function buat cek dataset"""
    
    print("🔍 MULAI INSPEKSI DATASET...")
    
    # 1. Cek struktur folder
    print("\n📂 STRUKTUR FOLDER:")
    print("-"*70)
    if os.path.exists(DATA_PATH):
        cek_struktur_folder(DATA_PATH)
    else:
        print(f"❌ Folder data gak ketemu di: {DATA_PATH}")
        
        # Coba cek alternative path
        alt_paths = [
            "/kaggle/working/data",
            "/kaggle/input/datasets/yahyaazz/dataset-skripsi/penyakit daun padi"
        ]
        
        for alt in alt_paths:
            if os.path.exists(alt):
                print(f"✅ Tapi nemu folder di: {alt}")
                print(f"   Isinya: {os.listdir(alt)[:5]}")
    
    # 2. Hitung statistik
    detail = hitung_statistik_dataset(TRAIN_PATH, TEST_PATH)
    
    if detail:
        # 3. Preview random images dari beberapa kelas
        print("\n\n👀 PREVIEW RANDOM IMAGES:")
        print("-"*70)
        
        # Pilih 3 kelas random buat di-preview
        kelas_list = list(detail.keys())
        if kelas_list:
            preview_kelas = np.random.choice(kelas_list, min(3, len(kelas_list)), replace=False)
            
            for kelas in preview_kelas:
                train_class_path = os.path.join(TRAIN_PATH, kelas)
                if os.path.exists(train_class_path):
                    print(f"\n🖼️  Preview kelas: {kelas}")
                    preview_random_images(train_class_path, num_samples=4, class_name=kelas)
    
    # 4. Kesimpulan
    print("\n" + "="*70)
    print("📝 KESIMPULAN:")
    print("="*70)
    
    if detail:
        total_gambar = sum(v['total'] for v in detail.values())
        print(f"✅ Dataset berhasil di-split dengan {len(detail)} kelas dan {total_gambar} gambar!")
        print(f"🎯 Siap buat training model CNN!")
        
        if total_gambar < 1000:
            print("⚠️  Total gambar kurang dari 1000, saran: pake data augmentation biar makin jos!")
        elif total_gambar < 5000:
            print("👍 Ukuran dataset cukup, bisa langsung training!")
        else:
            print("🔥 Wow! Dataset lo gede banget! Siap buat model powerfull!")
    else:
        print("❌ Ada yang error nih. Cek lagi path atau jalanin ulang script split!")
    
    print("\n💡 Next step: Lo mau lanjut preprocessing atau langsung bikin model?")
    print("="*70)

# ===== JALANKAN INSPEKTUR =====
cek_dataset_komplit()

# Bonus: Cek distribusi kelas pake bar chart
def plot_distribusi_kelas(detail_data):
    """Plot distribusi kelas"""
    if not detail_data:
        return
    
    kelas = list(detail_data.keys())
    train_counts = [detail_data[k]['train'] for k in kelas]
    test_counts = [detail_data[k]['test'] for k in kelas]
    
    x = np.arange(len(kelas))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(15, 6))
    bars1 = ax.bar(x - width/2, train_counts, width, label='Train', color='skyblue')
    bars2 = ax.bar(x + width/2, test_counts, width, label='Test', color='lightcoral')
    
    ax.set_xlabel('Kelas Penyakit')
    ax.set_ylabel('Jumlah Gambar')
    ax.set_title('Distribusi Dataset Train vs Test per Kelas')
    ax.set_xticks(x)
    ax.set_xticklabels(kelas, rotation=45, ha='right')
    ax.legend()
    
    # Tambah nilai di atas bar
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{int(height)}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Jalanin plot kalo detail ada
if 'detail' in locals() and detail:
    plot_distribusi_kelas(detail)